# Reddit Dataset Cleaning Pipeline
## *From Tweets to Trades: Analysing Sentiment Signal and Information Network for Crypto Market Prediction*

Cleans Reddit dataset CSV files (one per asset class) by removing spam, bots, deleted posts, low-quality content, and posts irrelevant to the target asset. Each file is processed and saved independently, preserving the per-asset structure for separate sentiment analysis.

**Workflow:**
1. Upload all per-asset Reddit CSVs at once.
2. Run cleaning on each file automatically.
3. Download all cleaned files as a ZIP.

## 0. Setup & Dependencies

In [ ]:
!pip install -q pandas

import pandas as pd
import re
import os
import json
import shutil
from datetime import datetime
from pathlib import Path
from google.colab import files

os.makedirs('data/raw/reddit', exist_ok=True)
os.makedirs('data/processed/reddit', exist_ok=True)
os.makedirs('data/processed/reddit/stats', exist_ok=True)
print('Setup complete.')

Setup complete.


## 1. Configuration

Defines the study period, spam keywords, bot patterns, and asset-specific relevance keywords. Each Reddit file gets matched to an asset based on its filename, and the appropriate keywords are used for relevance filtering.

In [ ]:
# --- Study period ---
START_DATE = pd.Timestamp('2020-01-01')
END_DATE = pd.Timestamp('2024-12-31')

# --- Length thresholds (Reddit posts can be longer than tweets) ---
MIN_TEXT_LENGTH = 15
MAX_TEXT_LENGTH = 10000

# --- Spam keywords specific to Reddit crypto spam patterns ---
SPAM_KEYWORDS = [
    'join my discord', 'join our discord', 'telegram group',
    'join my channel', 'join our channel', 'referral code',
    'referral link', 'use my code', 'sign up bonus', 'sign up using',
    'limited time offer', 'guaranteed profit', 'guaranteed returns',
    '100x gem', '1000x potential', 'next moonshot', 'next 1000x',
    'pump signal', 'premium signals', 'vip group', 'vip signals',
    'join vip', 'dm for signals', 'dm me for', 'send me 1 btc',
    'send me eth', 'private key', 'wallet hack', 'free airdrop',
    'claim airdrop', 'claim your free', 'click the link in my bio',
    'link in bio', 'follow my profile', 'f4f', 'follow back',
    'upvote bot', 'shilling my',
]

# --- Bot patterns (price tickers, repost bots, etc.) ---
BOT_PATTERNS = [
    r'^[A-Z]{3,5}\s*[:|]\s*\$[\d,\.]+',          # Price update bot
    r'^(\s*u/\w+\s*){2,}$',                       # Pure user mention spam
    r'^(\s*r/\w+\s*){3,}$',                       # Pure subreddit spam
    r'(.)\1{8,}',                                  # Repeated character spam
    r'^[\|\+\-=\s]+$',                            # ASCII art tables
    r'i am a bot,\s*and this action was performed', # Reddit bot signature
]

# --- Asset-specific relevance keywords ---
ASSET_KEYWORDS = {
    'bitcoin':   ['bitcoin', 'btc', 'satoshi'],
    'ethereum':  ['ethereum', 'ether', 'eth', 'vitalik'],
    'dogecoin':  ['dogecoin', 'doge'],
    'shiba_inu': ['shiba', 'shib', 'shibarmy', 'shiba inu'],
    'tether':    ['tether', 'usdt', 'stablecoin'],
    'general':   ['crypto', 'cryptocurrency', 'bitcoin', 'ethereum',
                  'blockchain', 'defi', 'altcoin'],
}

print(f'Study period: {START_DATE.date()} to {END_DATE.date()}')
print(f'Asset categories: {list(ASSET_KEYWORDS.keys())}')

Study period: 2020-01-01 to 2024-12-31
Asset categories: ['bitcoin', 'ethereum', 'dogecoin', 'shiba_inu', 'tether', 'general']


In [ ]:
# --- Helper functions ---

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&lt;', '<', text)
    text = re.sub(r'&gt;', '>', text)
    return re.sub(r'\s+', ' ', text).strip()

def is_deleted_or_removed(text):
    if not text:
        return True
    text_clean = text.strip().lower()
    deleted_markers = [
        '[deleted]', '[removed]', '[deleted by user]',
        '[removed by reddit]', '[ removed ]', '[ deleted ]',
        'deleted by user', 'removed by moderator',
    ]
    return (any(m in text_clean for m in deleted_markers) or
            text_clean in ('[deleted]', '[removed]', ''))

def is_spam(text):
    if not text:
        return True
    text_lower = text.lower()
    return any(kw in text_lower for kw in SPAM_KEYWORDS)

def is_bot_pattern(text):
    if not text:
        return True
    return any(re.search(p, text, re.IGNORECASE) for p in BOT_PATTERNS)

def is_relevant(text, keywords):
    if not text or not keywords:
        return False
    text_lower = text.lower()
    return any(kw in text_lower for kw in keywords)

def detect_asset_from_filename(filename):
    fname_lower = filename.lower()
    # Check specific asset names first.
    for asset in ['shiba_inu', 'shibainu', 'shibarmy', 'shib',
                  'bitcoin', 'ethereum', 'dogecoin', 'tether']:
        if asset in fname_lower:
            # Map shorthand variations back to the canonical key.
            if asset in ('shibainu', 'shibarmy', 'shib'):
                return 'shiba_inu'
            return asset
    # Fallback for general subreddit files.
    if any(g in fname_lower for g in ['general', 'cryptocurrency',
                                       'satoshistreetbets',
                                       'cryptomarkets']):
        return 'general'
    return None

def detect_columns(df):
    cols = {col.lower(): col for col in df.columns}
    text_col = next((cols[c] for c in ['text','selftext','body','content','post_text']
                     if c in cols), None)
    title_col = next((cols[c] for c in ['title','post_title']
                      if c in cols), None)
    date_col = next((cols[c] for c in ['date','created_utc','created_at','timestamp']
                     if c in cols), None)
    return text_col, title_col, date_col

print('Helper functions loaded.')

Helper functions loaded.


## 2. Upload All Reddit CSVs at Once

Use the upload button below to select **all** your per-asset Reddit CSV files. You can hold Shift or Cmd/Ctrl to select multiple files at once. Files will be saved into `data/raw/reddit/`.

In [ ]:
print('Upload all your Reddit dataset CSVs (one per asset class):\n')
uploaded = files.upload()

for fname, content in uploaded.items():
    dest = f'data/raw/reddit/{fname}'
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'  Saved: {dest} ({len(content)/1e6:.1f} MB)')

print(f'\n{len(uploaded)} file(s) uploaded.')

Upload all your Reddit dataset CSVs (one per asset class):



Saving bitcoin_posts.csv to bitcoin_posts.csv
Saving dogecoin_posts.csv to dogecoin_posts.csv
Saving ethereum_posts.csv to ethereum_posts.csv
Saving shiba_inu_posts.csv to shiba_inu_posts.csv
  Saved: data/raw/reddit/bitcoin_posts.csv (21.8 MB)
  Saved: data/raw/reddit/dogecoin_posts.csv (10.5 MB)
  Saved: data/raw/reddit/ethereum_posts.csv (23.5 MB)
  Saved: data/raw/reddit/shiba_inu_posts.csv (14.0 MB)

4 file(s) uploaded.


## 3. Cleaning Function

Applies all filters in sequence: deleted/removed posts, date range, length, spam, bot patterns, asset relevance, and exact-text deduplication. Each file is processed independently to preserve the per-asset structure.

In [ ]:
def clean_reddit_file(input_path, output_path, stats_path, asset_keywords):
    print('\n' + '=' * 60)
    print(f'CLEANING: {os.path.basename(input_path)}')
    print('=' * 60)

    try:
        df = pd.read_csv(input_path, on_bad_lines='skip',
                         engine='python', encoding='utf-8',
                         encoding_errors='ignore')
    except Exception as e:
        print(f'  ERROR loading file: {e}')
        return None

    initial = len(df)
    print(f'  Loaded: {initial:,} posts')

    text_col, title_col, date_col = detect_columns(df)
    print(f'  Text: {text_col} | Title: {title_col} | Date: {date_col}')

    if not text_col and not title_col:
        print(f'  ERROR: No text or title column. Cols: {list(df.columns)}')
        return None

    # Combine title and body for filtering.
    if title_col and text_col:
        df['_combined'] = (df[title_col].fillna('').astype(str) + ' ' +
                           df[text_col].fillna('').astype(str)).apply(clean_text)
    elif title_col:
        df['_combined'] = df[title_col].fillna('').astype(str).apply(clean_text)
    else:
        df['_combined'] = df[text_col].fillna('').astype(str).apply(clean_text)

    stats = {k: 0 for k in ['initial','dropped_deleted','dropped_date',
                             'dropped_length','dropped_spam','dropped_bot',
                             'dropped_irrelevant','dropped_duplicate','kept']}
    stats['initial'] = initial

    # 1. Drop deleted/removed posts.
    b = len(df); df = df[~df['_combined'].apply(is_deleted_or_removed)]
    stats['dropped_deleted'] = b - len(df)

    # 2. Date range.
    if date_col:
        try:
            parsed = pd.to_datetime(df[date_col], errors='coerce', utc=True)
            df['_date'] = parsed.dt.tz_localize(None)
            b = len(df)
            df = df[(df['_date'] >= START_DATE) & (df['_date'] <= END_DATE)]
            stats['dropped_date'] = b - len(df)
        except Exception as e:
            print(f'  Warning: date filter failed ({e})')

    # 3. Length.
    b = len(df); lens = df['_combined'].str.len()
    df = df[(lens >= MIN_TEXT_LENGTH) & (lens <= MAX_TEXT_LENGTH)]
    stats['dropped_length'] = b - len(df)

    # 4. Spam.
    b = len(df); df = df[~df['_combined'].apply(is_spam)]
    stats['dropped_spam'] = b - len(df)

    # 5. Bot patterns.
    b = len(df); df = df[~df['_combined'].apply(is_bot_pattern)]
    stats['dropped_bot'] = b - len(df)

    # 6. Relevance.
    b = len(df)
    df = df[df['_combined'].apply(lambda t: is_relevant(t, asset_keywords))]
    stats['dropped_irrelevant'] = b - len(df)

    # 7. Deduplicate.
    b = len(df); df = df.drop_duplicates(subset=['_combined'], keep='first')
    stats['dropped_duplicate'] = b - len(df)

    # Drop helper columns.
    df = df.drop(columns=[c for c in ['_combined','_date'] if c in df.columns])
    stats['kept'] = len(df)

    df.to_csv(output_path, index=False)

    print(f'\n  --- Summary ---')
    for key, val in stats.items():
        print(f'  {key:25s} {val:>10,}')
    if initial > 0:
        print(f'  {"retention":25s} {100 * stats["kept"] / initial:>9.1f}%')
    print(f'  Saved: {output_path}')

    if stats_path:
        with open(stats_path, 'w') as f:
            f.write(f'Reddit Cleaning Statistics\nFile: {input_path}\n')
            f.write(f'Run: {datetime.now().isoformat()}\n' + '=' * 50 + '\n\n')
            for key, val in stats.items():
                f.write(f'{key}: {val:,}\n')

    return stats

print('Cleaning function loaded.')

Cleaning function loaded.


## 4. Process All Uploaded Files

Iterates through every Reddit CSV uploaded in Step 2, automatically detects which asset each one corresponds to from its filename, and runs the cleaning pipeline on each. Results are saved to `data/processed/reddit/`.

In [ ]:
INPUT_DIR = 'data/raw/reddit'
OUTPUT_DIR = 'data/processed/reddit'
STATS_DIR = 'data/processed/reddit/stats'

csv_files = sorted([f for f in os.listdir(INPUT_DIR)
                    if f.endswith('.csv') and 'all_reddit' not in f.lower()])

if not csv_files:
    print(f'No CSV files found in {INPUT_DIR}. Upload files in Step 2 first.')
else:
    print('=' * 60)
    print(f'PROCESSING {len(csv_files)} REDDIT FILE(S)')
    print('=' * 60)
    for f in csv_files:
        size_mb = os.path.getsize(os.path.join(INPUT_DIR, f)) / 1e6
        print(f'  {f} ({size_mb:.1f} MB)')

    overall_stats = {}
    for fname in csv_files:
        input_path = os.path.join(INPUT_DIR, fname)
        output_path = os.path.join(OUTPUT_DIR, fname.replace('.csv', '_cleaned.csv'))
        stats_path = os.path.join(STATS_DIR, fname.replace('.csv', '_stats.txt'))

        asset = detect_asset_from_filename(fname)
        if not asset:
            print(f'\n  Note: could not infer asset for {fname}, '
                  f'using general crypto keywords.')
            asset = 'general'
        keywords = ASSET_KEYWORDS.get(asset, ASSET_KEYWORDS['general'])
        print(f'\n  Detected asset: {asset}')
        print(f'  Keywords: {keywords}')

        stats = clean_reddit_file(input_path, output_path, stats_path, keywords)
        if stats:
            overall_stats[fname] = stats

    # Final summary.
    print('\n' + '=' * 60)
    print('OVERALL SUMMARY')
    print('=' * 60)
    total_initial = sum(s['initial'] for s in overall_stats.values())
    total_kept = sum(s['kept'] for s in overall_stats.values())
    print(f'Files processed:    {len(overall_stats)}')
    print(f'Total input posts:  {total_initial:,}')
    print(f'Total kept posts:   {total_kept:,}')
    if total_initial > 0:
        print(f'Overall retention:  {100 * total_kept / total_initial:.1f}%')

    print(f'\nPer-file results:')
    for fname, stats in overall_stats.items():
        ret = 100 * stats['kept'] / stats['initial'] if stats['initial'] else 0
        print(f'  {fname:50s} {stats["kept"]:>8,} kept ({ret:5.1f}%)')

PROCESSING 4 REDDIT FILE(S)
  bitcoin_posts.csv (21.8 MB)
  dogecoin_posts.csv (10.5 MB)
  ethereum_posts.csv (23.5 MB)
  shiba_inu_posts.csv (14.0 MB)

  Detected asset: bitcoin
  Keywords: ['bitcoin', 'btc', 'satoshi']

CLEANING: bitcoin_posts.csv
  Loaded: 49,766 posts
  Text: selftext | Title: title | Date: date

  --- Summary ---
  initial                       49,766
  dropped_deleted               25,395
  dropped_date                       0
  dropped_length                   886
  dropped_spam                     245
  dropped_bot                       49
  dropped_irrelevant             9,880
  dropped_duplicate                221
  kept                          13,090
  retention                      26.3%
  Saved: data/processed/reddit/bitcoin_posts_cleaned.csv

  Detected asset: dogecoin
  Keywords: ['dogecoin', 'doge']

CLEANING: dogecoin_posts.csv
  Loaded: 35,067 posts
  Text: selftext | Title: title | Date: date

  --- Summary ---
  initial                       35,067

## 5. Download All Cleaned Files

Bundles all cleaned CSVs and stats into a single ZIP for download.

In [ ]:
zip_name = 'reddit_cleaned'
shutil.make_archive(zip_name, 'zip', 'data/processed', 'reddit')
zip_path = f'{zip_name}.zip'
size_mb = os.path.getsize(zip_path) / 1e6
print(f'Archive: {zip_path} ({size_mb:.1f} MB)')
files.download(zip_path)
print('Download started.')

Archive: reddit_cleaned.zip (9.4 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started.
